### pair-specs dataset

In [12]:
import itertools
import random

roles = [
    "A person",
    "A scientist",
    "A professor",
    "An engineer",
    "A doctor",
    "A teacher",
    "A student",
    "A writer",
    "A photographer",
    "A chef",
    "An artist",
    "A musician",
    "A librarian",
    "A researcher",
    "A traveler",
    "A programmer",
    "A designer",
    "A journalist",
    "A nurse",
    "An architect",
    "A barista",
    "A botanist",
    "A historian",
    "A mathematician",
    "A physicist",
    "A biologist",
    "A chemist",
    "A geologist",
    "A pilot",
    "A firefighter",
    "A dancer",
    "A singer",
    "A filmmaker",
    "A coach",
    "A swimmer",
    "A hiker",
    "A gamer",
    "A carpenter",
    "A mechanic",
    "A gardener",
    "A baker",
    "A tailor",
    "A calligrapher",
    "A poet",
    "A translator",
    "A curator",
    "A consultant",
    "A lawyer",
    "An entrepreneur",
    "A marketer",
]

activities = [
    "reading a book",
    "using a laptop",
    "writing notes",
    "drinking coffee",
    "giving a lecture",
    "analyzing data",
    "studying",
    "sketching",
    "taking photos",
    "cooking",
    "painting",
    "playing the guitar",
    "browsing shelves",
    "conducting an experiment",
    "packing a bag",
    "debugging code",
    "drawing a diagram",
    "interviewing someone",
    "taking a patient history",
    "reviewing blueprints",
    "making espresso",
    "examining a plant",
    "presenting a talk",
    "solving equations",
    "testing a hypothesis",
    "looking through a microscope",
    "mixing chemicals",
    "reading a map",
    "checking instruments",
    "holding a fire hose",
    "practicing choreography",
    "singing on stage",
    "editing a film",
    "coaching a team",
    "swimming laps",
    "tying hiking boots",
    "playing a strategy game",
    "measuring wood",
    "fixing an engine",
    "watering plants",
    "kneading dough",
    "sewing fabric",
    "writing calligraphy",
    "reciting a poem",
    "translating a paragraph",
    "arranging exhibits",
    "advising a client",
    "arguing a case",
    "pitching an idea",
    "planning a campaign",
]

locations = [
    "in a library",
    "in a classroom",
    "in a laboratory",
    "in a cafe",
    "at home",
    "in an office",
    "in a coworking space",
    "in a studio",
    "on a stage",
    "in a kitchen",
    "in a museum",
    "in a park",
    "on a train",
    "at the airport",
    "by the seaside",
    "on a balcony",
    "in a workshop",
    "in a greenhouse",
    "in a conference room",
    "in a lecture hall",
    "in a bookstore",
    "on a rooftop",
    "in a courtyard",
    "in a quiet study room",
    "in a bustling market",
]

random.seed(42)  # deterministic order for reproducibility

pairs = []
for r, a, l in itertools.product(roles, activities, locations):
    base = f"{r} {a} {l}"
    with_specs = f"{r} wearing spectacles {a} {l}"
    pairs.append((base, with_specs))
    if len(pairs) == 1000:
        break

prompt_pairs = pairs
len(prompt_pairs)

1000

In [13]:
# pip install -U datasets huggingface_hub
import os

from datasets import Dataset, DatasetDict, Features, Value
from huggingface_hub import HfApi, create_repo

features = Features(
    {
        "pair_index": Value("int32"),
        "base_prompt": Value("string"),
        "with_spectacles_prompt": Value("string"),
    }
)

data = {
    "pair_index": list(range(len(prompt_pairs))),
    "base_prompt": [a for a, _ in prompt_pairs],
    "with_spectacles_prompt": [b for _, b in prompt_pairs],
}

ds = Dataset.from_dict(data, features=features).shuffle(seed=42)

# 3) ---- Split train/test (e.g., 80/20) ----
splits = ds.train_test_split(test_size=0.2, seed=42)
dset = DatasetDict(
    {
        "train": splits["train"],
        "test": splits["test"],
    }
)

# 4) ---- Push to the Hub ----
# Fill these in:
HF_USERNAME = "nirmalendu01"
REPO_NAME = "spectacles-bias-prompts"  # change if you like
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"

# Create the repo if it doesn't exist yet (set private=True if you want)
try:
    create_repo(REPO_ID, repo_type="dataset", private=False, exist_ok=True)
except Exception as e:
    print("Note:", e)

# 5) ---- (Optional) Add a dataset card ----
readme = f"""\
# {REPO_NAME}

A minimal prompt-pair dataset for spectacles bias studies in text-to-image diffusion.

## Contents
- `base_prompt`: the neutral prompt
- `with_spectacles_prompt`: the same prompt with "wearing spectacles"
- `pair_index`: stable pairing id

## Splits
- train: {len(dset["train"])} examples
- test:  {len(dset["test"])} examples

## Intended use
- Measure latent deltas (with − without) on UNet outputs
- Learn/text-side debias directions; cross-attention analyses

## License
- Specify your license here (e.g., CC-BY-4.0)
"""
open("README.md", "w", encoding="utf-8").write(readme)

# Upload the README (optional but nice)
api = HfApi()
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card",
)

# 6) ---- Push the DatasetDict ----
dset.push_to_hub(REPO_ID, private=False, commit_message="Initial upload: train/test splits")

print(f"Done. Dataset pushed to: https://huggingface.co/datasets/{REPO_ID}")

/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/huggingface_hub/hf_api.py:9706: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


KeyboardInterrupt: 

### combine with bigger dataset - total =10k

In [ ]:
# pip install -U datasets huggingface_hub
import itertools
import json
import random

from datasets import Dataset, DatasetDict, Features, Value, load_dataset
from huggingface_hub import HfApi, create_repo

# ----------------------------
# CONFIG — CHANGE THESE
# ----------------------------
HF_USERNAME = "nirmalendu01"
REPO_NAME = "text2image-10k-with-spectacles-pairs"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
PRIVATE = False

SEED = 42
random.seed(SEED)

N_BASE = 9000  # rows from jackyhate/text-to-image-2M (go into text column)
N_TOTAL = 10000  # final dataset size
N_SPECS = N_TOTAL - N_BASE  # 1000 rows will come from your pairs (base/with specs as separate rows)

# `prompt_pairs` must exist in scope:
# prompt_pairs = [(base_prompt, with_spectacles_prompt), ...]  # ~100 pairs

# ----------------------------
# 0) AUTH (choose ONE)
# ----------------------------
# Option A: preset token: os.environ["HF_TOKEN"] = "hf_***"
# Option B: interactive:
# login()  # uncomment if you want an interactive prompt

# ----------------------------
# 1) Load 9k prompts from the public dataset (streaming)
# ----------------------------
streaming_ds = load_dataset("jackyhate/text-to-image-2M", split="train", streaming=True)


def extract_text(example):
    """
    Robustly extract a text field:
    - If there's a 'json' string, try to parse and use its 'text' key if present
    - Else use the first string field we find
    """
    if "json" in example and isinstance(example["json"], dict):
        # try:
        #     j = json.loads(example["json"])
        # common keys to try for prompt text
        j = ex["json"]
        for k in ("text", "prompt", "caption", "content"):
            if k in j and isinstance(j[k], str):
                return j[k]
        # except Exception:
        #     pass
    # fallback: first string field
    for k, v in example.items():
        if isinstance(v, str):
            return v
    raise KeyError("Could not find a text field in example.")


base_prompts = []
for ex in streaming_ds:
    try:
        txt = extract_text(ex)
        if txt and txt.strip():
            base_prompts.append(txt.strip())
    except Exception:
        continue
    if len(base_prompts) >= N_BASE:
        break

if len(base_prompts) < N_BASE:
    raise RuntimeError(
        f"Only collected {len(base_prompts)} base prompts; increase streaming range or check extractor."
    )

# Optional: light shuffle for variety (spacing will be preserved later)
random.shuffle(base_prompts)

# ----------------------------
# 2) Flatten your pairs into N_SPECS individual rows
#    (both 'base' and 'with spectacles' become separate rows in `text`)
# ----------------------------
if "prompt_pairs" not in globals():
    raise RuntimeError("You must define `prompt_pairs` (list of (base, with_spectacles)).")

if len(prompt_pairs) == 0:
    raise ValueError("`prompt_pairs` is empty.")

# Flatten: [base_1, with_1, base_2, with_2, ...]
flat_pairs = list(itertools.chain.from_iterable(prompt_pairs))  # length = 2 * len(pairs)

# Upsample (cycle) to exactly N_SPECS rows
spec_rows = list(itertools.islice(itertools.cycle(flat_pairs), N_SPECS))

# Optional: shuffle the order of spectacles rows so they’re not paired back-to-back
random.shuffle(spec_rows)

# ----------------------------
# 3) Interleave equally: place one spectacles row every `gap` positions
# ----------------------------
gap = N_TOTAL // N_SPECS  # e.g., 10000 // 1000 = 10
combined = [None] * N_TOTAL

spec_i = 0
for pos in range(0, N_TOTAL, gap):
    if spec_i >= N_SPECS:
        break
    combined[pos] = {
        "text": spec_rows[spec_i],
        "is_from_pair": True,
        "has_spectacles_phrase": ("wearing spectacles" in spec_rows[spec_i].lower()),
        "source": "user_specs",
    }
    spec_i += 1

# Fill remaining slots with base prompts
base_iter = iter(base_prompts)
for i in range(N_TOTAL):
    if combined[i] is None:
        try:
            p = next(base_iter)
        except StopIteration:
            # Shouldn't happen, but pad if needed
            p = random.choice(base_prompts)
        combined[i] = {
            "text": p,
            "is_from_pair": False,
            "has_spectacles_phrase": False,
            "source": "jackyhate/text-to-image-2M",
        }

# Sanity checks
assert len(combined) == N_TOTAL
assert sum(1 for r in combined if r["is_from_pair"]) == N_SPECS

# ----------------------------
# 4) Create DatasetDict with 80/20 split
#    IMPORTANT: shuffle=False to keep equal spacing intact in each split
# ----------------------------
features = Features(
    {
        "text": Value("string"),
        "is_from_pair": Value("bool"),
        "has_spectacles_phrase": Value("bool"),
        "source": Value("string"),
    }
)

ds_all = Dataset.from_list(combined, features=features)

# Keep order; do not shuffle before split if you want spacing preserved.
splits = ds_all.train_test_split(test_size=0.2, seed=SEED, shuffle=False)
dset = DatasetDict({"train": splits["train"], "test": splits["test"]})

print(dset)
print(
    "Counts:",
    "train",
    len(dset["train"]),
    "test",
    len(dset["test"]),
    "spectacles train",
    sum(dset["train"]["is_from_pair"]),
    "spectacles test",
    sum(dset["test"]["is_from_pair"]),
)

# ----------------------------
# 5) Push to the Hub
# ----------------------------
# Create repo if needed
create_repo(REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)

readme = f"""\
# {REPO_NAME}

A 10k **text-only** dataset combining:
- **9,000** prompts sampled (streaming) from `jackyhate/text-to-image-2M`
- **1,000** rows drawn from user-provided *spectacles* **pairs** (both base and with "wearing spectacles" versions are included as separate rows)

## Schema
- `text`: the prompt
- `is_from_pair`: whether this row came from the spectacles pairs
- `has_spectacles_phrase`: whether the text explicitly includes "wearing spectacles"
- `source`: `"jackyhate/text-to-image-2M"` or `"user_specs"`

## Construction
- Spectacles rows are **equally spaced** in the 10k corpus (one every ~{gap} rows).
- 80/20 split performed **without shuffling** to preserve spacing.

## Intended Use
- Bias/steering analyses for T2I diffusion, cross-attention & UNet activation studies.

## License
- Base data inherits license of `jackyhate/text-to-image-2M`.
- User spectacles prompts: specify your license here.
"""
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

api = HfApi()
api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
    commit_message="Add dataset card",
)

# Push datasets
dset.push_to_hub(
    REPO_ID, private=PRIVATE, commit_message="10k text dataset with equally spaced spectacles rows"
)
print(f"Done. Dataset pushed to: https://huggingface.co/datasets/{REPO_ID}")

### fairface

In [ ]:
# make_fairface_hf_dataset.py
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict, Image
from huggingface_hub import create_repo

# -------- CONFIG --------
ROOT = Path("fairface-img-margin025-trainval")  # contains train/ and val/
TRAIN_CSV = Path("fairface_label_train.csv")
VAL_CSV = Path("fairface_label_val.csv")

# If your CSV's filename column is known, set it:
IMAGE_FILENAME_COL = None  # e.g. "file"
REPO_ID = "nirmalendu01/fairface-trainval"  # change me
PRIVATE = False
HF_TOKEN = None  # or string token; else relies on CLI login
# ------------------------

COMMON_NAME_GUESSES = [
    "file",
    "image_file",
    "image",
    "filename",
    "name",
    "path",
    "filepath",
    "relpath",
]


def detect_image_col(df: pd.DataFrame) -> str:
    if IMAGE_FILENAME_COL:
        if IMAGE_FILENAME_COL not in df.columns:
            raise ValueError(
                f"IMAGE_FILENAME_COL='{IMAGE_FILENAME_COL}' not in CSV columns: {list(df.columns)}"
            )
        return IMAGE_FILENAME_COL
    for c in COMMON_NAME_GUESSES:
        if c in df.columns:
            return c
    # Heuristic: pick a string column containing typical image extensions
    for c in df.columns:
        if (
            df[c].dtype == "object"
            and df[c]
            .astype(str)
            .str.contains(r"\.(jpg|jpeg|png|webp)$", case=False, regex=True)
            .any()
        ):
            return c
    raise ValueError("Could not find an image-filename column. Set IMAGE_FILENAME_COL explicitly.")


def normalize_relpath(s: str) -> str:
    # trim whitespace and leading "./"
    s = str(s).strip().lstrip("./")
    return s


def build_split(csv_path: Path, img_root: Path) -> Dataset:
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    if not img_root.exists():
        raise FileNotFoundError(f"Image dir not found: {img_root}")

    df = pd.read_csv(csv_path)
    img_col = detect_image_col(df)

    # Ensure a fresh "image" column is CREATED
    # If CSV already stores subfolders (e.g., "train/xxx.jpg"), don't double-prepend.
    def make_abs(p_str: str) -> str:
        rel = normalize_relpath(p_str)
        rel_path = Path(rel)
        if rel_path.is_absolute():
            return str(rel_path)
        # If it already starts with the split dir name (train/ or val/), don't add again
        if len(rel_path.parts) > 0 and rel_path.parts[0] in {"train", "val"}:
            full = ROOT / rel_path
        else:
            full = img_root / rel_path
        return str(full.resolve())

    df["image"] = df[img_col].astype(str).map(make_abs)

    # Optional: drop rows whose files are missing (prevents ZeroDivisionError later)
    exists = df["image"].map(lambda p: Path(p).exists())
    n_missing = (~exists).sum()
    if n_missing:
        print(f"[WARN] {n_missing} rows in {csv_path} reference missing files; dropping them.")
        df = df.loc[exists].reset_index(drop=True)

    # Build datasets and ensure correct Image() feature
    ds = Dataset.from_pandas(df, preserve_index=False)
    ds = ds.cast_column("image", Image())
    return ds


train_dir = ROOT / "train"
val_dir = ROOT / "val"

train_ds = build_split(TRAIN_CSV, train_dir)
val_ds = build_split(VAL_CSV, val_dir)

# Keep only non-empty splits (avoids size-estimation crash on empty)
non_empty = {k: v for k, v in {"train": train_ds, "val": val_ds}.items() if len(v) > 0}
if not non_empty:
    raise ValueError(
        "Both splits are empty after path checks. Verify filename column and image locations."
    )
dropped = set({"train", "val"}) - set(non_empty.keys())
if dropped:
    print(f"[WARN] Dropping empty splits: {sorted(dropped)}")

dsd = DatasetDict(non_empty)

# Optional: quick sanity print
for split, ds in dsd.items():
    print(f"{split}: {len(ds)} rows, columns={ds.column_names}")
    print(ds[:2])  # show first 2

create_repo(REPO_ID, repo_type="dataset", private=PRIVATE, exist_ok=True)

print(f"Pushing to Hub -> {REPO_ID}")
dsd.push_to_hub(REPO_ID)
print("Done.")

train: 86744 rows, columns=['file', 'age', 'gender', 'race', 'service_test', 'image']
{'file': ['train/1.jpg', 'train/2.jpg'], 'age': ['50-59', '30-39'], 'gender': ['Male', 'Female'], 'race': ['East Asian', 'Indian'], 'service_test': [True, False], 'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=224x224 at 0x7F5F88F41E90>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=224x224 at 0x7F5F88F42750>]}
val: 10954 rows, columns=['file', 'age', 'gender', 'race', 'service_test', 'image']
{'file': ['val/1.jpg', 'val/2.jpg'], 'age': ['3-9', '50-59'], 'gender': ['Male', 'Female'], 'race': ['East Asian', 'East Asian'], 'service_test': [False, True], 'image': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=224x224 at 0x7F5F88F42710>, <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=224x224 at 0x7F5F88F41590>]}


HfHubHTTPError: (Request ID: Root=1-68ef420a-4ccc6e9d4848cd9d75944fea;3f590621-badf-4662-a716-e7c932e3afa0)

403 Forbidden: You don't have the rights to create a dataset under the namespace "your-username".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

### sub-sample fairface and create new datasets

In [ ]:
# pip install datasets huggingface_hub
import itertools

from datasets import ClassLabel, Dataset, DatasetDict, concatenate_datasets, load_dataset
from huggingface_hub import HfApi


def take_k_per_class(ds, column: str, k: int, seed: int = 42) -> Dataset:
    """Return up to k samples per unique value in `column`, preserving all other columns (incl. image)."""
    # Ensure the image column (if present) stays as Image() feature
    if "image" in ds.column_names and not isinstance(ds.features["image"], Image):
        ds = ds.cast_column("image", Image())

    # Work out the label values
    feat = ds.features.get(column)
    if isinstance(feat, ClassLabel):
        labels = list(range(feat.num_classes))  # ints

        # filter uses ints already
        def is_label(ex, v):
            return ex[column] == v
    else:
        # fall back to the set of observed values (strings or ints)
        labels = sorted(set(ds[column]))

        def is_label(ex, v):
            return ex[column] == v

    per_class = []
    for v in labels:
        dsv = ds.filter(is_label, fn_kwargs={"v": v})
        n = min(k, len(dsv))
        if n == 0:
            continue
        dsv = dsv.shuffle(seed=seed).select(range(n))
        per_class.append(dsv)

    if not per_class:
        # empty result; return empty dataset with same features
        return ds.select([])

    out = concatenate_datasets(per_class).shuffle(seed=seed)
    return out


def build_and_push_balanced(
    source_repo: str = "nirmalendu01/fairface-trainval",
    column: str = "race",
    k_train: int = 100,
    k_val: int = 50,
    *,
    target_repo: str = "your-username/fairface-trainval-balanced-k",
    private: bool = False,
    seed: int = 42,
):
    # 1) load both splits
    ds = load_dataset(source_repo)  # expects {"train", "val"}
    assert "train" in ds and "val" in ds, "Dataset must have 'train' and 'val' splits."

    # 2) take k per class in each split independently
    train_bal = take_k_per_class(ds["train"], column, k_train, seed)
    val_bal = take_k_per_class(ds["val"], column, k_val, seed)

    out = DatasetDict({"train": train_bal, "val": val_bal})

    # 3) (optional) print class counts sanity check
    def counts(d):
        # simple tally
        from collections import Counter

        return Counter(d[column])

    print("train counts:", counts(train_bal))
    print("val counts:", counts(val_bal))

    # 4) push to hub (creates repo if it doesn’t exist)
    out.push_to_hub(target_repo, private=private)
    print(f"Pushed to https://huggingface.co/datasets/{target_repo}")

(0, 0)

In [ ]:
# -------- run it --------
# Example: pick k=200 per race for both train & val, and push to your namespace
build_and_push_balanced(
    source_repo="nirmalendu01/fairface-trainval",
    column="race",
    k_train=200,
    k_val=50,
    target_repo="nirmalendu01/fairface-trainval-race-balanced-200",
    private=False,
    seed=123,
)

### sample from socialcounterfactual

In [ ]:
races = {
    "white": 0,
    "black": 1,
    "asian": 2,
    "indian": 3,
    "latino": 4,
    "Middle Eastern": 5,
    "none": 6,
}
genders = {"male": 0, "female": 1, "none": 2}

In [23]:
import random
from collections import defaultdict

from datasets import load_dataset

# ---- Config ----
random.seed(42)

# Map your buckets → dataset race label strings (case-insensitive match)
target_map = {
    "white": "white",
    "black": "black",
    "asian": "asian",  # dataset uses "east asian"
    "indian": "indian",
    "latino": "latino",  # dataset uses "latino-hispanic"
    "middle eastern": "middle eastern",  # likely absent; kept for completeness
}
TARGETS = list(target_map.keys())
K = 250  # samples per label


# ---- Helpers ----
def norm(x):
    return str(x).strip().lower() if x is not None else ""


def is_race_row(ex) -> bool:
    return norm(ex.get("dataset_segment")) == "race_gender"


def pick_race_label(ex) -> str | None:
    """
    Return our canonical target key ('asian', 'latino', ...) if the row's race label matches.
    Prefer a1 if it's 'race', else fall back to a2 if it's 'race'.
    """
    # Prefer a1 if it's race
    if norm(ex.get("a1_type")) == "race":
        lab = norm(ex.get("a1"))
    elif norm(ex.get("a2_type")) == "race":
        lab = norm(ex.get("a2"))
    else:
        return None

    for k, ds_name in target_map.items():
        if lab == norm(ds_name):
            return k
    return None


def reservoir_add(buf, total_seen, item, k=K):
    """Reservoir sample into buf with capacity k, given we've seen total_seen items of this label."""
    if len(buf) < k:
        buf.append(item)
    else:
        j = random.randint(0, total_seen - 1)
        if j < k:
            buf[j] = item


# ---- Streaming + sampling ----
reservoir = {t: [] for t in TARGETS}
seen_counts = dict.fromkeys(TARGETS, 0)

ds_iter = load_dataset("Intel/SocialCounterfactuals", split="train", streaming=True)

total = 0
for ex in ds_iter:
    total += 1
    print("streamed rows:", total, end="\r")
    if not is_race_row(ex):
        continue

    key = pick_race_label(ex)
    if key is None:
        continue

    seen_counts[key] += 1

    # Store only lightweight fields to avoid pulling image bytes
    item = {
        "caption": ex.get("caption"),
        "a1": ex.get("a1"),
        "a2": ex.get("a2"),
        "a1_type": ex.get("a1_type"),
        "a2_type": ex.get("a2_type"),
        "a1a2": ex.get("a1a2"),
        # "counterfactual_set": ex.get("counterfactual_set"),
    }
    reservoir_add(reservoir[key], seen_counts[key], item, k=K)

    # Optional early-exit when all buckets are full
    if all(len(reservoir[t]) >= K for t in TARGETS):
        break

for t in TARGETS:
    print(f"{t:>15}: kept {len(reservoir[t]):4d}  | seen {seen_counts[t]:6d}")

# Example: your 200 random samples for "asian"
samples_asian = reservoir["asian"]
samples_asian[:2]

          white: kept  250  | seen    250
          black: kept  250  | seen    250
          asian: kept  250  | seen    250
         indian: kept  250  | seen    250
         latino: kept  250  | seen    250
 middle eastern: kept  250  | seen    250


[{'caption': 'A photo of an Asian male Accountant',
  'a1': 'Asian',
  'a2': 'male',
  'a1_type': 'race',
  'a2_type': 'gender',
  'a1a2': 'Asian male'},
 {'caption': 'A photo of an Asian female Accountant',
  'a1': 'Asian',
  'a2': 'female',
  'a1_type': 'race',
  'a2_type': 'gender',
  'a1a2': 'Asian female'}]

### add none samples

In [29]:
# ds_iter = load_dataset("nirmalendu01/PromptBiasDB", split="train", streaming=True)

# total = 0
# for ex in ds_iter:
#     total += 1
#     print("streamed rows:", total, end="\r")

import random

from datasets import load_dataset


def norm_axes(v):
    if v is None:
        return None
    if isinstance(v, list):
        return [str(x).strip().lower() for x in v]
    if isinstance(v, dict) and "axes" in v:
        a = v["axes"]
        return [str(x).strip().lower() for x in a] if isinstance(a, list) else None
    if isinstance(v, str):
        s = v.strip()
        try:
            obj = json.loads(s)
            return norm_axes(obj)
        except Exception:
            parts = [p.strip().lower() for p in s.replace(",", " ").split() if p.strip()]
            return parts or None
    return None


def sample_k_none_streaming(repo_id: str, split="train", k: int = 250, seed: int = 42):
    random.seed(seed)
    ds_iter = load_dataset(repo_id, split=split, streaming=True)

    buf = []  # reservoir
    seen = 0  # how many qualifying rows seen

    for ex in ds_iter:
        axes = norm_axes(ex.get("bias_axes"))
        if axes == ["none"]:
            seen += 1
            if len(buf) < k:
                buf.append(ex)
            else:
                j = random.randint(1, seen)
                if j <= k:
                    buf[j - 1] = ex
    return buf  # list of dicts (rows)


# Example:
rows = sample_k_none_streaming("nirmalendu01/PromptBiasDB", split="train", k=250)
print(len(rows), rows[:2])

250 [{'prompt': 'Detailed face  close-up  ultra HD', 'bias_axes': ['none'], 'id': 'pbdb-0130513'}, {'prompt': 'a sad clown with missing legs', 'bias_axes': ['none'], 'id': 'pbdb-0023357'}]


### combine both and upload to hf

In [30]:
from datasets import Dataset, DatasetDict, Features, Value
from huggingface_hub import HfApi

In [31]:
# --- normalize into one simple schema: id, caption, a1, a2, a1_type, a2_type, a1a2 ---
def build_rows(rows_rg, rows_p):
    out = []

    # 1) keep your race/gender rows as-is, ensure id exists (create a simple one if missing)
    for i, r in enumerate(rows_rg):
        out.append(
            {
                "id": r.get("id") or f"rg-{i:06d}",
                "caption": r.get("caption"),
                "a1": r.get("a1"),
                "a2": r.get("a2"),
                "a1_type": r.get("a1_type"),
                "a2_type": r.get("a2_type"),
                "a1a2": r.get("a1a2"),
            }
        )

    # 2) convert prompt→caption; everything else None
    for j, r in enumerate(rows_p):
        out.append(
            {
                "id": r.get("id") or f"pbdb-{j:06d}",
                "caption": r.get("prompt"),
                "a1": None,
                "a2": None,
                "a1_type": None,
                "a2_type": None,
                "a1a2": None,
            }
        )

    return out


rows_race_gender = [item for k, v in reservoir.items() for item in v]
rows_prompts = rows

combined = build_rows(rows_race_gender, rows_prompts)

# --- make a small HF dataset (single split) ---
features = Features(
    {
        "id": Value("string"),
        "caption": Value("string"),
        "a1": Value("string"),
        "a2": Value("string"),
        "a1_type": Value("string"),
        "a2_type": Value("string"),
        "a1a2": Value("string"),
    }
)
ds = Dataset.from_list(combined).cast(features)

# 200 each race for
dd = DatasetDict({"train": ds})  # one split; change if you want val/test too

Casting the dataset: 100%|██████████| 1750/1750 [00:00<00:00, 653143.98 examples/s]


In [32]:
# save locally (optional)
dd.save_to_disk("fairface_promptbiasdb_combined")

Saving the dataset (1/1 shards): 100%|██████████| 1750/1750 [00:00<00:00, 464323.89 examples/s]


In [33]:
len(dd["train"])

1750

In [20]:
# load back (optional)
dd = DatasetDict.load_from_disk("fairface_promptbiasdb_combined")

In [34]:
import random
from collections import Counter

from datasets import DatasetDict

random.seed(42)


def _norm(x):
    return x.strip().lower() if isinstance(x, str) else None


def _race_from_row(ex):
    # Prefer a1 if it's a race; else a2 if it's a race.
    if _norm(ex.get("a1_type")) == "race" and ex.get("a1") is not None:
        return _norm(ex["a1"])
    if _norm(ex.get("a2_type")) == "race" and ex.get("a2") is not None:
        return _norm(ex["a2"])
    return None  # treat as "none"


ds_all = dd["train"]

# Group indices by race; collect "none" separately
idxs_by_race = defaultdict(list)
none_idxs = []
for i, ex in enumerate(ds_all):
    r = _race_from_row(ex)
    if r is None:
        none_idxs.append(i)
    else:
        idxs_by_race[r].append(i)

# For each race, take exactly 200 → train and 50 → val (requires ≥250 rows)
train_idxs, val_idxs = [], []
for race, idxs in idxs_by_race.items():
    rng = random.Random(42 ^ (hash(race) & 0xFFFFFFFF))
    rng.shuffle(idxs)
    if len(idxs) < 250:
        print(f"[WARN] race '{race}' has only {len(idxs)} rows; need ≥250. Skipping this race.")
        continue
    train_idxs.extend(idxs[:200])
    val_idxs.extend(idxs[200:250])

# Now add "none" group: 200 to train, 50 to val (requires ≥250)
rng_none = random.Random(1337)
rng_none.shuffle(none_idxs)
if len(none_idxs) < 250:
    print(
        f"[WARN] 'none' group has only {len(none_idxs)} rows; need ≥250. Skipping 'none' sampling."
    )
else:
    train_idxs.extend(none_idxs[:200])
    val_idxs.extend(none_idxs[200:250])

# Build new splits
ds_train = ds_all.select(train_idxs)
ds_val = ds_all.select(val_idxs)
dd = DatasetDict({"train": ds_train, "validation": ds_val})
print(ds_train, ds_val)


# Optional: show counts per race (including 'none') in each split
def count_races(ds):
    c = Counter()
    for ex in ds:
        r = _race_from_row(ex)
        c[r if r is not None else "none"] += 1
    return dict(c)


print("Train counts:", count_races(ds_train))
print("Val counts:  ", count_races(ds_val))
print(dd)

Dataset({
    features: ['id', 'caption', 'a1', 'a2', 'a1_type', 'a2_type', 'a1a2'],
    num_rows: 1400
}) Dataset({
    features: ['id', 'caption', 'a1', 'a2', 'a1_type', 'a2_type', 'a1a2'],
    num_rows: 350
})
Train counts: {'white': 200, 'black': 200, 'asian': 200, 'indian': 200, 'latino': 200, 'middle eastern': 200, 'none': 200}
Val counts:   {'white': 50, 'black': 50, 'asian': 50, 'indian': 50, 'latino': 50, 'middle eastern': 50, 'none': 50}
DatasetDict({
    train: Dataset({
        features: ['id', 'caption', 'a1', 'a2', 'a1_type', 'a2_type', 'a1a2'],
        num_rows: 1400
    })
    validation: Dataset({
        features: ['id', 'caption', 'a1', 'a2', 'a1_type', 'a2_type', 'a1a2'],
        num_rows: 350
    })
})


In [35]:
# --- push to the Hub ---
DST_REPO = "nirmalendu01/socialcounterfactuals-200"  # change as you like
PRIVATE = True  # set False to make public

api = HfApi(token=os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN"))
api.create_repo(repo_id=DST_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
dd.push_to_hub(DST_REPO, private=PRIVATE)

print(f"✅ Uploaded {len(dd)} rows to https://huggingface.co/datasets/{DST_REPO}")

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 53.09ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 23.4kB / 23.4kB, 14.6kB/s  


Processing Files (1 / 1)                : 100%|██████████| 23.4kB / 23.4kB, 11.7kB/s  
New Data Upload                         : 100%|██████████| 23.4kB / 23.4kB, 11.7kB/s  
                                        : 100%|██████████| 23.4kB / 23.4kB            
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 66.90ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████| 8.48kB / 8.48kB, 8.48kB/s  


Processing Files (1 / 1)                : 100%|██████████| 8.48kB / 8.48kB, 6.05kB/s  
New Data Upload                         : 100%|██████████| 8.48kB / 8.48kB, 6.05kB/s  
                                        : 100%|████████

✅ Uploaded 2 rows to https://huggingface.co/datasets/nirmalendu01/socialcounterfactuals-200


### count gender - male and female in dataset and balance

In [36]:
import re


# Normalize to 'male' / 'female' (extend synonyms as needed)
def _norm_gender(x: str | None) -> str | None:
    if not isinstance(x, str):
        return None
    s = x.strip().lower()
    # simple lemmatization of common variants
    male_syn = {"male"}
    fem_syn = {"female"}
    # handle punctuation/plurals
    s = re.sub(r"[^\w\s]", "", s)
    if s in male_syn:
        return "male"
    if s in fem_syn:
        return "female"
    return None


def count_gender_in_dataset(ds) -> tuple[dict[str, int], dict[str, int]]:
    """
    Returns (occurrence_counts, row_counts) for one HF Dataset.
    - occurrence_counts: counts every time a1/a2 has a gender label
    - row_counts: counts a row once per gender even if both a1 and a2 match
    """
    occ = Counter()
    row = Counter()
    for ex in ds:
        genders_in_row = set()
        for col in ("a1", "a2"):
            label = ex.get(col, None)
            typ = ex.get(f"{col}_type", None)
            if isinstance(typ, str) and typ.strip().lower() == "gender":
                g = _norm_gender(label)
                if g is not None:
                    occ[g] += 1
                    genders_in_row.add(g)
        for g in genders_in_row:
            row[g] += 1
    return dict(occ), dict(row)


# --- Use with your DatasetDict dd (e.g., dd = DatasetDict({"train": ds})) ---


def count_gender_across_splits(dd):
    totals_occ = Counter()
    totals_row = Counter()
    per_split = {}

    for split, ds in dd.items():
        occ, row = count_gender_in_dataset(ds)
        per_split[split] = {"occurrences": occ, "rows": row}
        totals_occ.update(occ)
        totals_row.update(row)

    per_split["TOTAL"] = {"occurrences": dict(totals_occ), "rows": dict(totals_row)}
    return per_split


# Example:
result = count_gender_across_splits(dd)
print(
    result["train"]
)  # {'occurrences': {'male': X, 'female': Y}, 'rows': {'male': A, 'female': B}}
print(result["TOTAL"])  # aggregated over all splits

{'occurrences': {'female': 596, 'male': 604}, 'rows': {'female': 596, 'male': 604}}
{'occurrences': {'female': 750, 'male': 750}, 'rows': {'female': 750, 'male': 750}}


### convert null to None

In [ ]:
import math

from datasets import DatasetDict, load_dataset

# -------- helpers --------
NULL_STRINGS = {"", "null", "none", "nan"}


def _normalize_scalar(x):
    # floats: NaN -> None
    if isinstance(x, float) and math.isnan(x):
        return None
    # strings: common null tokens -> None
    if isinstance(x, str) and x.strip().lower() in NULL_STRINGS:
        return None
    # leave other types as-is (int/bool/None/etc.)
    return x


def _normalize_any(x):
    if isinstance(x, dict):
        return {k: _normalize_any(v) for k, v in x.items()}
    if isinstance(x, list):
        return [_normalize_any(v) for v in x]
    return _normalize_scalar(x)


def _clean_example(example):
    # map() with batched=False will pass a single dict per row
    return _normalize_any(example)


# -------- load, clean, (optional) save/push --------
# Loads all splits if present
dd = load_dataset("nirmalendu01/socialcounterfactuals-200")

# NOTE: If your dataset is large, you can set num_proc>1 for parallelism.
cleaned = DatasetDict({split: ds.map(_clean_example, batched=False) for split, ds in dd.items()})

# (optional) save to disk
# cleaned.save_to_disk("./socialcounterfactuals-200-cleaned")

# (optional) push to hub under a new repo
cleaned.push_to_hub("nirmalendu01/socialcounterfactuals-200")

### remove a1,a2,a1_type,a2_type and a1a2 columns

In [ ]:
from typing import Any

from datasets import DatasetDict, load_dataset


# ---- optional normalization helpers ----
def _norm_str(x: str | None) -> str | None:
    if not isinstance(x, str):
        return None
    s = x.strip()
    return s if s else None


def _norm_gender(x: str | None) -> str | None:
    if not isinstance(x, str):
        return None
    s = re.sub(r"[^\w\s]", "", x.strip().lower())
    if s in {"male", "man", "boy", "m", "masculine"}:
        return "male"
    if s in {"female", "woman", "girl", "f", "feminine"}:
        return "female"
    return _norm_str(x)  # fall back to original (kept as-is)


def _norm_race(x: str | None) -> str | None:
    # keep original label but strip whitespace; customize if you have a canonical set
    return _norm_str(x)


def _pick(first: str | None, second: str | None) -> str | None:
    """Prefer first non-null; else second; ignore conflicts."""
    return first if first is not None else second


# ---- row transform ----
def _extract_race_gender(ex: dict[str, Any]) -> dict[str, Any]:
    race = None
    gender = None

    a1, a2 = ex.get("a1"), ex.get("a2")
    t1, t2 = ex.get("a1_type"), ex.get("a2_type")

    t1 = (t1 or "").strip().lower() if isinstance(t1, str) else None
    t2 = (t2 or "").strip().lower() if isinstance(t2, str) else None

    # From a1
    if t1 == "race":
        race = _norm_race(a1)
    elif t1 == "gender":
        gender = _norm_gender(a1)

    # From a2 (fill missing; ignore conflicts if both present)
    if t2 == "race":
        race = _pick(race, _norm_race(a2))
    elif t2 == "gender":
        gender = _pick(gender, _norm_gender(a2))

    conflict = (
        t1 == "race"
        and t2 == "race"
        and _norm_race(a1)
        and _norm_race(a2)
        and _norm_race(a1) != _norm_race(a2)
    )
    if conflict:
        print(f"[WARN] conflicting race labels in row id={ex.get('id')}: a1='{a1}' vs a2='{a2}'")
    return {"race": race, "gender": gender, "conflict": bool(conflict)}

    # return {"race": race, "gender": gender}


# ---- apply to your dataset and drop old columns ----
dd = load_dataset("nirmalendu01/socialcounterfactuals-200")

cols_to_drop = ["a1", "a2", "a1_type", "a2_type", "a1a2"]

cleaned = DatasetDict(
    {
        split: ds.map(
            _extract_race_gender,
            batched=False,
            remove_columns=[
                c for c in cols_to_drop if c in ds.column_names
            ],  # drop only if present
        )
        for split, ds in dd.items()
    }
)

# (Optional) verify a few rows
print(cleaned)
print(cleaned[next(iter(cleaned.keys()))][0])

# (Optional) save/push
# cleaned.save_to_disk("./socialcounterfactuals-200-race-gender")
cleaned.push_to_hub("nirmalendu01/socialcounterfactuals-200")

Map: 100%|██████████| 350/350 [00:00<00:00, 19826.14 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'caption', 'race', 'gender', 'conflict'],
        num_rows: 1400
    })
    validation: Dataset({
        features: ['id', 'caption', 'race', 'gender', 'conflict'],
        num_rows: 350
    })
})
{'id': 'rg-000111', 'caption': 'A photo of a White female bartender', 'race': 'White', 'gender': 'female', 'conflict': False}
